# Store Sales Forecasting (Retail) with LSTM

## Objectives
Forecast **daily sales** using an LSTM on a public time-series CSV (demand proxy).

## Theory
Retail **demand forecasting** uses past sales windows to predict future units/revenue. LSTMs capture weekly seasonality and trends better than plain moving averages.

**Loss:** MSE on scaled target. **Metric:** RMSE / MAE in original sales units after inverse transform.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Problem Definition & Business Context
Stores optimize **inventory**, staffing, and promotions from demand forecasts. Under-forecasting causes stockouts; over-forecasting wastes capital.


In [ ]:
# Store sales — Kaggle Rossmann requires login; use retail sales time series sample
url = "https://raw.githubusercontent.com/plotly/datasets/master/time-series-19-covid-combined.csv"
df = pd.read_csv(url)
# Use US daily cases as proxy univariate sales-like series for teaching
if "US" in df.columns:
    ts = df[["Date","US"]].dropna().rename(columns={"Date":"date","US":"sales"})
else:
    ts = df.iloc[:, :2]
    ts.columns = ["date","sales"]
ts["date"] = pd.to_datetime(ts["date"])
ts = ts.set_index("date").sort_index()
ts.plot(figsize=(12,3))
plt.title("Retail demand proxy series")
plt.show()

# EDA — rolling mean
ts["roll7"] = ts["sales"].rolling(7).mean()
ts[["sales","roll7"]].plot(figsize=(12,3))
plt.title("Sales vs 7-day rolling mean")
plt.show()


In [ ]:
values = ts["sales"].values.astype(np.float32)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(values.reshape(-1,1)).flatten()
lookback = 14

def make_seq(s, lb):
    X, y = [], []
    for i in range(lb, len(s)):
        X.append(s[i-lb:i])
        y.append(s[i])
    return np.array(X), np.array(y)

X, y = make_seq(scaled, lookback)
X = X[..., np.newaxis]
split = int(0.8*len(X))
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]
val_split = int(0.9 * len(X_train))
X_tr, y_tr = X_train[:val_split], y_train[:val_split]
X_val, y_val = X_train[val_split:], y_train[val_split:]
print("Train", X_tr.shape, "Val", X_val.shape, "Test", X_test.shape)


In [ ]:
model = models.Sequential([
    layers.LSTM(48, input_shape=(lookback,1)),
    layers.Dropout(0.2),
    layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()


In [ ]:
store_cb = [
    callbacks.ModelCheckpoint("retail_store_lstm_best.keras", save_best_only=True),
    callbacks.EarlyStopping(patience=6, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
]
history = model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=40, batch_size=32,
                    callbacks=store_cb, verbose=1)
pd.DataFrame(history.history)[["loss","val_loss"]].plot()
plt.show()


In [ ]:
pred_scaled = model.predict(X_test, verbose=0)
pred = scaler.inverse_transform(pred_scaled)
actual = scaler.inverse_transform(y_test.reshape(-1,1))
rmse = np.sqrt(mean_squared_error(actual, pred))
mae = mean_absolute_error(actual, pred)
print(f"Test RMSE: {rmse:.2f} MAE: {mae:.2f}")
plt.figure(figsize=(12,4))
plt.plot(actual, label="actual")
plt.plot(pred, label="forecast")
plt.legend()
plt.title("Retail store sales — test forecast")
plt.show()


In [ ]:
# Inference: forecast next day from last window
last_window = scaled[-lookback:].reshape(1, lookback, 1)
next_scaled = model.predict(last_window, verbose=0)[0,0]
next_sale = scaler.inverse_transform([[next_scaled]])[0,0]
print(f"Next-day sales forecast: {next_sale:.2f}")
model.save("retail_store_lstm.keras")
import joblib
joblib.dump(scaler, "retail_sales_scaler.pkl")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
